# DeepImpact (2021)

---
[[paper]](https://arxiv.org/pdf/2108.06782)<br>DeepImpact = Deep Passage Impact

DeepImpact – это метод Dense Retrieval, предложенный исследователями из Microsoft (Oleg Ivanov, Ameya Prabhu, Sean MacAvaney) в 2021 году. Он представляет собой нейросетевой подход к информационному поиску, который учится присваивать **оценки важности (impact scores)** отдельным токенам в документе, используя глубокую нейронную сеть, но при этом сохраняет индексацию и механизм поиска, характерные для Sparse Retrieval систем.

### Контекст

Традиционные методы информационного поиска, такие как BM25 (поздние 1990-е), основываются на подсчете совпадений ключевых слов (Sparse Retrieval). Они очень эффективны и легко интерпретируемы, но плохо справляются с синонимами, полисемией и общим семантическим смыслом запроса или документа.

С появлением больших языковых моделей (LLM) стали популярны методы Dense Retrieval (например, DPR (2020), ColBERT (2020)). Эти модели кодируют запросы и документы в плотные векторные представления (embeddings) и измеряют их сходство (например, с помощью косинусного сходства или скалярного произведения). Они прекрасно улавливают семантическое сходство, но могут быть менее эффективны для точных совпадений ключевых слов, особенно для редких терминов, и требуют специализированных методов индексации (например, ANN - Approximate Nearest Neighbor) для больших корпусов, которые могут быть ресурсоемкими.

### Идея метода

Основная идея DeepImpact заключается в том, чтобы **объединить сильные стороны Sparse и Dense Retrieval**. Вместо того чтобы кодировать весь документ в один плотный вектор или полагаться только на статистику слов, DeepImpact учится присваивать **контекстуализированные оценки важности (impact scores)** каждому отдельному токену в документе. Эти оценки указывают, насколько важен данный токен в контексте всего документа для его релевантности потенциальному запросу. Эти scores затем используются в механизме поиска, напоминающем традиционный инвертированный индекс.

### Постановка задачи

Решается стандартная задача информационного поиска: по заданному запросу $Q$ найти $K$ наиболее релевантных документов из большой коллекции $D = \{d_1, d_2, \ldots, d_N\}$. Релевантность определяется тем, насколько хорошо документ отвечает на запрос или соответствует его смыслу.

### Существующие альтернативные методы

На момент появления DeepImpact существовали следующие основные подходы:

*   **Sparse Retrieval (например, BM25)**: Основан на частоте слов и их распределении.
    *   **Архитектурное отличие**: Не использует нейронные сети для понимания смысла. Релевантность определяется статистическими формулами на основе совпадения токенов.
    *   **Проблемы**: Плохо обрабатывает синонимы и семантическое сходство.
*   **Dense Retrieval (например, DPR (2020))**: Кодирует запросы и документы в плотные векторы с помощью двух отдельных BERT-подобных энкодеров и вычисляет сходство векторов.
    *   **Архитектурное отличие**: Документы кодируются в один глобальный вектор, теряется информация о важности отдельных токенов. Индексация требует ANN.
    *   **Проблемы**: Может пропустить точные совпадения по редким терминам.
*   **Late Interaction Models (например, ColBERT (2020))**: Создает отдельные векторы для каждого токена в запросе и документе, а затем вычисляет их попарное сходство на этапе инференса для агрегирования итоговой оценки.
    *   **Архитектурное отличие**: Документы все еще хранятся как наборы векторов токенов, а не как скалярные "impact scores", что делает индексацию и поиск более сложными, чем в DeepImpact.
*   **Pre-computed Impact Models (например, DeepCT (2020))**: Предшественник DeepImpact, который использовал BERT для предсказания **терминовой важности (term importance)** для каждого токена. Однако DeepCT предсказывал бинарное значение (важен/не важен) или вес, но не встраивал это так явно в традиционный Sparse Retrieval фреймворк с инвертированными индексами, как DeepImpact.
    *   **Архитектурное отличие**: DeepImpact совершенствует эту идею, делая impact scores непосредственно частью ранжирования по инвертированному индексу и обучая их более целенаправленно для ранжирования.

Новизна DeepImpact заключается в том, что он **учится присваивать скалярные, контекстуализированные impact scores каждому токену документа с помощью глубокой нейронной сети и использует их в эффективной структуре инвертированного индекса**. Это позволяет получить семантическую мощь нейросетей при сохранении эффективности и прозрачности Sparse Retrieval.

### Архитектура модели

Архитектура DeepImpact относительно проста и состоит из следующих основных компонентов:

1.  **Базовый Энкодер (Base Encoder)**:
    *   Это стандартная Transformer-модель, такая как BERT или RoBERTa.
    *   На вход энкодеру подается документ (passage).
    *   На выходе для каждого токена документа генерируется его **контекстуализированное векторное представление**.

2.  **Impact Head**:
    *   Это небольшой линейный слой (или однослойный MLP), который применяется к векторному представлению каждого токена, полученному от Базового Энкодера.
    *   Impact Head преобразует векторное представление токена в **единственное скалярное значение** — его **impact score**.
    *   Формально, для каждого токена $t$ в документе $P$, имеющего векторное представление $h_t$ от Базового Энкодера, impact score вычисляется как $s_t = \text{Linear}(h_t)$.

3.  **Функция ранжирования (Scoring Function)**:
    *   Для данного запроса $Q$ и документа $P$, итоговая оценка релевантности $\text{score}(Q, P)$ рассчитывается как **сумма impact scores** всех токенов документа $P$, которые также присутствуют в запросе $Q$.
    *   $\text{score}(Q, P) = \sum_{t \in Q \cap P} s_t$, где $s_t$ — это impact score токена $t$ в документе $P$.
    *   Могут использоваться модификации, например, учет важности токенов запроса (аналогично IDF), но базовая идея – сумма impact scores.

Таким образом, DeepImpact генерирует для каждого токена документа не просто его векторное представление, а скалярную величину, которая напрямую используется в механизме ранжирования, подобном BM25.

### Алгоритм обучения

DeepImpact обучается на парах (запрос, релевантный документ), аналогично другим методам Retrieval.

1.  **Формирование обучающих данных**:
    *   Используются наборы данных, содержащие запросы и соответствующие им релевантные документы (позитивные примеры), а также нерелевантные документы (негативные примеры).
    *   Критически важно использовать **жесткие негативные примеры (hard negatives)** – документы, которые изначально кажутся релевантными (например, содержат много ключевых слов запроса), но на самом деле не отвечают на запрос. Это заставляет модель учиться более тонким семантическим нюансам и предсказывать impact scores, которые лучше различают истинно релевантные документы от ложных.
    *   Хард-негативы могут быть получены из других моделей Retrieval (например, BM25, DPR) или путем майнинга из больших коллекций.

2.  **Loss-функция**:
    *   Обучение обычно строится на **ранжирующей Loss-функции**, такой как **Negative Log-Likelihood (NLL)** или **Triplet Loss**.
    *   При использовании NLL, модель пытается максимизировать вероятность того, что позитивный документ будет иметь более высокую оценку, чем негативные документы, среди всех кандидатов для данного запроса.
    *   Например, для запроса $Q$, позитивного документа $D^+$ и набора негативных документов $D^- = \{D^-_1, \ldots, D^-_k\}$, Loss-функция может быть:
        $L = -\log \left( \frac{\exp(\text{score}(Q, D^+))}{\sum_{D_j \in \{D^+\} \cup D^-} \exp(\text{score}(Q, D_j))} \right)$
    *   Эта функция заставляет модель увеличивать impact scores релевантных токенов в позитивных документах и уменьшать их в негативных, чтобы позитивные документы получали более высокую итоговую оценку.

3.  **Оптимизация**:
    *   Модель обучается с использованием стандартных оптимизаторов (например, AdamW) для настройки весов Базового Энкодера и Impact Head.

### Алгоритм индексации

После обучения модели, для эффективного поиска по большой коллекции документов, необходимо построить специализированный индекс:

1.  **Обработка документов**:
    *   Для каждого документа $P$ в коллекции:
        *   Документ пропускается через обученный Базовый Энкодер и Impact Head.
        *   Для каждого токена $t$ в документе $P$ вычисляется его **impact score** $s_t$.

2.  **Построение инвертированного индекса (Inverted Index)**:
    *   Создается инвертированный индекс, аналогичный тому, что используется в Sparse Retrieval.
    *   Однако, вместо того чтобы просто хранить список документов, содержащих токен, для каждого токена $t$ в индексе хранится список пар `(document_id, impact_score)`, где `impact_score` — это вычисленная оценка важности токена $t$ в соответствующем документе `document_id`.
    *   Пример: для токена "neural" в индексе может храниться `[(doc1, 0.7), (doc5, 0.9), (doc10, 0.2)]`.

### Алгоритм инференса (поиска)

Когда поступает новый запрос $Q$, процесс поиска происходит следующим образом:

1.  **Токенизация запроса**:
    *   Запрос $Q$ токенизируется на отдельные термины $q_1, q_2, \ldots, q_m$.

2.  **Поиск по инвертированному индексу**:
    *   Для каждого токена запроса $q_i$:
        *   Выполняется поиск по инвертированному индексу.
        *   Извлекаются все документы, содержащие $q_i$, вместе с их соответствующими impact scores для этого токена: `(document_id, impact_score)`.

3.  **Агрегирование оценок**:
    *   Для каждого уникального документа, найденного на предыдущем шаге, его итоговая оценка релевантности $\text{score}(Q, P)$ вычисляется как **сумма всех impact scores** его токенов, которые совпадают с токенами запроса.
    *   Например, если запрос "neural network" и документ $P$ содержит оба слова с impact scores 0.7 и 0.9 соответственно, то $\text{score}(Q, P) = 0.7 + 0.9 = 1.6$.

4.  **Ранжирование и выдача результатов**:
    *   Документы ранжируются по убыванию их итоговых оценок.
    *   Возвращаются $K$ документов с наивысшими оценками.

### Результаты

DeepImpact демонстрирует превосходные результаты на различных стандартных бенчмарках информационного поиска, таких как MS MARCO Passage Ranking.

*   **Значительное улучшение по сравнению со Sparse Retrieval (BM25)**: DeepImpact, как правило, превосходит BM25 на 15-25 процентных пунктов по метрикам ранжирования, таким как MRR (Mean Reciprocal Rank) или nDCG (Normalized Discounted Cumulative Gain) на задачах, требующих семантического понимания. Например, на MS MARCO Passage Ranking DeepImpact может показать MRR@10 до 35-40%, тогда как BM25 обычно около 18-20%.
*   **Конкурентоспособность с Dense Retrieval**: DeepImpact показывает производительность, сравнимую или даже превосходящую чисто Dense Retrieval методы, такие как DPR (2020), особенно в сценариях, где точное совпадение терминов также играет важную роль.
*   **Эффективность**: Благодаря использованию инвертированного индекса, DeepImpact сохраняет вычислительную эффективность Sparse Retrieval методов, что делает его привлекательным для использования в крупномасштабных поисковых системах по сравнению с методами, требующими ANN-поиска по плотным векторам. Индексация и поиск выполняются быстрее и требуют меньше памяти, чем хранение и поиск по большим коллекциям плотных векторов.

## 📝 Критический анализ

```markdown
# DeepImpact (2021)

---
[[paper]](https://arxiv.org/pdf/2108.06782)<br>DeepImpact = Deep Passage Impact

DeepImpact — метод Dense Retrieval от Microsoft (Oleg Ivanov, Ameya Prabhu, Sean MacAvaney), который присваивает **оценки важности (impact scores)** токенам в документах, сохраняя индексацию Sparse Retrieval.

### Контекст

Традиционные методы, такие как BM25, эффективны, но плохо справляются с синонимами и семантикой. Dense Retrieval, например, DPR (2020) и ColBERT (2020), используют плотные векторы, но требуют ресурсоемкой индексации и могут пропускать точные совпадения.

### Идея

DeepImpact объединяет Sparse и Dense Retrieval, присваивая **контекстуализированные impact scores** токенам, что позволяет использовать инвертированный индекс.

### Задача

Поиск $K$ релевантных документов для запроса $Q$ из коллекции $D$.

### Альтернативы

- **Sparse Retrieval (BM25)**: Основан на частоте слов, не использует нейросети.
- **Dense Retrieval (DPR)**: Кодирует документы в плотные векторы, теряя информацию о токенах.
- **Late Interaction Models (ColBERT)**: Использует векторы токенов, усложняя индексацию.
- **Pre-computed Impact Models (DeepCT)**: Предсказывает важность токенов, но не интегрирует это в Sparse Retrieval.

### Архитектура

1. **Базовый Энкодер**: Transformer-модель (например, BERT), генерирует векторные представления токенов.
2. **Impact Head**: Преобразует вектор токена в **скалярный impact score**.
3. **Функция ранжирования**: Суммирует impact scores токенов, присутствующих в запросе и документе.

### Обучение

1. **Данные**: Пары (запрос, релевантный документ) с **жесткими негативными примерами**.
2. **Loss-функция**: Negative Log-Likelihood, максимизирует оценку релевантных документов.
3. **Оптимизация**: AdamW для настройки весов.

### Индексация

1. **Обработка документов**: Вычисление impact scores для токенов.
2. **Инвертированный индекс**: Хранит пары `(document_id, impact_score)` для токенов.

### Инференс

1. **Токенизация запроса**.
2. **Поиск по индексу**: Извлечение документов с impact scores.
3. **Агрегирование**: Сумма impact scores для совпадающих токенов.
4. **Ранжирование**: Возврат $K$ документов с наивысшими оценками.

### Результаты

DeepImpact превосходит BM25 на 15-25 п.п. по MRR и nDCG, достигая MRR@10 до 35-40% на MS MARCO. Он конкурентоспособен с Dense Retrieval и эффективен благодаря инвертированному индексу.

<img src="img/img.png" width=500>
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import numpy as np

# Define a simple Impact Head
class ImpactHead(nn.Module):
    def __init__(self, hidden_size):
        super(ImpactHead, self).__init__()
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)

# Initialize BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Initialize the Impact Head
impact_head = ImpactHead(hidden_size=model.config.hidden_size)

# Example document and query
document = "Deep learning models are powerful tools for data analysis."
query = "deep learning tools"

# Tokenize and encode the document
inputs = tokenizer(document, return_tensors='pt')
outputs = model(**inputs)

# Get the hidden states of the last layer
hidden_states = outputs.last_hidden_state

# Compute impact scores for each token in the document
impact_scores = impact_head(hidden_states)

# Tokenize the query
query_tokens = tokenizer.tokenize(query)

# Create a simple inverted index with impact scores
inverted_index = {}
for token, score in zip(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]), impact_scores.detach().numpy()):
    if token not in inverted_index:
        inverted_index[token] = []
    inverted_index[token].append(('doc1', score))

# Function to calculate relevance score for a query
def calculate_relevance_score(query_tokens, inverted_index):
    score = 0.0
    for token in query_tokens:
        if token in inverted_index:
            # Sum the impact scores for the tokens present in the query
            score += sum([entry[1] for entry in inverted_index[token]])
    return score

# Calculate the relevance score for the query
relevance_score = calculate_relevance_score(query_tokens, inverted_index)
print(f"Relevance score for the query '{query}': {relevance_score}")

# This is a simple illustration of how DeepImpact assigns impact scores to tokens
# and uses them in a sparse retrieval-like inverted index for efficient search.
# In a real-world scenario, you would train the model to optimize these scores
# using a ranking loss and a dataset with query-document pairs.
```

### Key Points Explained:

1. **BERT as Base Encoder**: We use a pre-trained BERT model to obtain contextualized embeddings for each token in the document. This is the first step in the DeepImpact architecture.

2. **Impact Head**: A simple linear layer (or MLP) that converts the BERT embeddings into scalar impact scores for each token. This is crucial for the DeepImpact method, as it allows us to retain the interpretability and efficiency of sparse retrieval methods.

3. **Inverted Index with Impact Scores**: We create an inverted index where each token maps to a list of documents and their corresponding impact scores. This structure is similar to traditional sparse retrieval methods but enhanced with learned scores.

4. **Relevance Scoring**: For a given query, we calculate the relevance score by summing the impact scores of the tokens present in both the query and the document. This mimics the ranking function used in DeepImpact.

5. **Efficiency and Interpretability**: By using impact scores in an inverted index, DeepImpact combines the semantic understanding of dense retrieval with the efficiency of sparse retrieval, making it suitable for large-scale search applications.